
# FinBERT No-News Day Audit

This notebook answers the two questions raised before submission:

1. How many no-news days occur in the **general**, **BTC-specific**, and **ETH-specific** FinBERT series?
2. Is `news_count` or a no-news indicator included in the forecasting inputs?

The audit follows the same article-level preparation logic used by the revised Phase 2–4 notebooks: exact UTC publication dates, duplicate removal, MarketWatch automated-recap filtering, and valid FinBERT-score filtering before daily aggregation. It does **not** alter any forecasting model.


In [1]:

import re
from pathlib import Path
from typing import Dict, Sequence

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Crypto_Research')
DATA_DIR = PROJECT_DIR / 'data'
NEWS_DIR = DATA_DIR / 'alphavantage_news'
OUTPUT_DIR = PROJECT_DIR / 'revised_outputs_v4' / 'no_news_audit'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETRIEVAL_START = pd.Timestamp('2022-04-01')
MODEL_START = pd.Timestamp('2022-05-01')
DATA_END = pd.Timestamp('2026-08-31')

PERIODS = {
    'full_retrieval_window': (pd.Timestamp('2022-04-01'), pd.Timestamp('2026-08-31')),
    'forecast_sample': (pd.Timestamp('2022-05-01'), pd.Timestamp('2026-08-31')),
    'train_query_dates': (pd.Timestamp('2022-05-01'), pd.Timestamp('2025-03-31')),
    'validation_query_dates': (pd.Timestamp('2025-04-01'), pd.Timestamp('2025-10-31')),
    'test_query_dates': (pd.Timestamp('2025-11-01'), pd.Timestamp('2026-08-31')),
}

FILES = {
    'general': NEWS_DIR / 'alphavantage_general_articles_finbert.csv',
    'BTC_specific': NEWS_DIR / 'alphavantage_BTC_crypto_articles_finbert.csv',
    'ETH_specific': NEWS_DIR / 'alphavantage_ETH_crypto_articles_finbert.csv',
}

FILTER_AUTOMATED_MARKET_RECAPS = True
MARKETWATCH_AUTOMATION_AUTHOR_TOKEN = 'marketwatch automation'
MARKETWATCH_RECAP_TITLE_REGEX = (
    r'\bstock\s+(?:outperforms|underperforms)\b.*(?:competitors|trading day)|'
    r'\bstock\s+(?:falls|rises|gains|slips|drops|climbs|advances|declines|rallies|surges|sinks|plunges)\b.*'
    r'(?:outperforms|underperforms)\s+(?:the\s+)?market\b'
)


Mounted at /content/drive


In [2]:

def normalize_title(value) -> str:
    text = '' if pd.isna(value) else str(value).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9 ]+', '', text)
    return text.strip()

def parse_published_at_utc(df: pd.DataFrame) -> pd.Series:
    for col in ['published_at_utc', 'time_published', 'published_at', 'datetime', 'timestamp']:
        if col in df.columns:
            values = df[col]
            if col == 'time_published':
                parsed = pd.to_datetime(values.astype(str), format='%Y%m%dT%H%M%S', errors='coerce', utc=True)
                fallback = pd.to_datetime(values, errors='coerce', utc=True)
                parsed = parsed.fillna(fallback)
            else:
                parsed = pd.to_datetime(values, errors='coerce', utc=True)
            return parsed
    if 'pub_date' in df.columns:
        return pd.to_datetime(df['pub_date'], errors='coerce', utc=True)
    raise ValueError('No supported publication timestamp column found.')

def filter_automated_marketwatch_recaps(df: pd.DataFrame) -> pd.DataFrame:
    if not FILTER_AUTOMATED_MARKET_RECAPS or df.empty:
        return df.copy()
    out = df.copy()
    source_col = next((c for c in ['source', 'source_name', 'publisher'] if c in out.columns), None)
    title_col = next((c for c in ['title', 'headline'] if c in out.columns), None)
    author_col = next((c for c in ['authors', 'author'] if c in out.columns), None)
    url_col = next((c for c in ['url', 'article_url', 'link'] if c in out.columns), None)

    is_marketwatch = pd.Series(False, index=out.index)
    if source_col:
        is_marketwatch = out[source_col].astype(str).str.contains('marketwatch', case=False, na=False)
    author_auto = pd.Series(False, index=out.index)
    if author_col:
        author_auto = out[author_col].astype(str).str.contains(MARKETWATCH_AUTOMATION_AUTHOR_TOKEN, case=False, na=False)
    title_template = pd.Series(False, index=out.index)
    if title_col:
        title_template = out[title_col].astype(str).str.contains(MARKETWATCH_RECAP_TITLE_REGEX, case=False, regex=True, na=False)
    data_news_url = pd.Series(False, index=out.index)
    if url_col:
        data_news_url = out[url_col].astype(str).str.contains('/data-news/', case=False, na=False)

    remove = is_marketwatch & (author_auto | (title_template & data_news_url))
    return out.loc[~remove].copy()

def prepare_articles(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing input file: {path}')
    df = pd.read_csv(path, low_memory=False)
    df['published_at_utc__audit'] = parse_published_at_utc(df)
    df = df[df['published_at_utc__audit'].notna()].copy()
    df['Date'] = df['published_at_utc__audit'].dt.tz_convert('UTC').dt.tz_localize(None).dt.normalize()
    df = df[(df['Date'] >= RETRIEVAL_START) & (df['Date'] <= DATA_END)].copy()
    df = filter_automated_marketwatch_recaps(df)

    # Exact/provider-ID deduplication when an identifier exists.
    id_col = next((c for c in ['article_id', 'id', 'uuid', 'news_id'] if c in df.columns), None)
    if id_col:
        nonempty = df[id_col].notna() & (df[id_col].astype(str).str.strip() != '')
        kept = df.loc[nonempty].drop_duplicates(subset=[id_col], keep='first')
        no_id = df.loc[~nonempty]
        df = pd.concat([kept, no_id], ignore_index=True)

    # Uniform modeling-stage normalized title + source + UTC publication-day deduplication.
    title_col = next((c for c in ['title', 'headline'] if c in df.columns), None)
    source_col = next((c for c in ['source', 'source_name', 'publisher'] if c in df.columns), None)
    if title_col and source_col:
        df['_title_norm'] = df[title_col].map(normalize_title)
        df['_source_norm'] = df[source_col].astype(str).str.lower().str.strip()
        valid_key = (df['_title_norm'] != '') & (df['_source_norm'] != '')
        keyed = df.loc[valid_key].drop_duplicates(subset=['_title_norm', '_source_norm', 'Date'], keep='first')
        unkeyed = df.loc[~valid_key]
        df = pd.concat([keyed, unkeyed], ignore_index=True)

    # Use the same practical FinBERT score fallbacks as the phase notebooks.
    score_col = next((c for c in [
        'finbert_sentiment_score', 'finbert_score', 'sentiment_score', 'score'
    ] if c in df.columns), None)
    if score_col is None:
        # If class probabilities are available, reconstruct pos - neg.
        pos_col = next((c for c in ['positive_probability', 'positive_prob', 'prob_positive'] if c in df.columns), None)
        neg_col = next((c for c in ['negative_probability', 'negative_prob', 'prob_negative'] if c in df.columns), None)
        if pos_col and neg_col:
            df['_audit_score'] = pd.to_numeric(df[pos_col], errors='coerce') - pd.to_numeric(df[neg_col], errors='coerce')
            score_col = '_audit_score'
        else:
            raise ValueError(f'Could not identify a FinBERT score column in {path.name}. Columns include: {list(df.columns)[:30]}')

    conf_col = next((c for c in [
        'finbert_confidence', 'confidence', 'sentiment_confidence'
    ] if c in df.columns), None)
    df['_score'] = pd.to_numeric(df[score_col], errors='coerce')
    if conf_col:
        df['_confidence'] = pd.to_numeric(df[conf_col], errors='coerce').fillna(1.0).clip(lower=1e-8)
    else:
        df['_confidence'] = 1.0
    return df[df['_score'].notna()].copy()

def daily_finbert(path: Path) -> pd.DataFrame:
    articles = prepare_articles(path)
    rows = []
    for date, group in articles.groupby('Date', sort=True):
        scores = group['_score'].to_numpy(float)
        conf = group['_confidence'].to_numpy(float)
        rows.append({
            'Date': pd.Timestamp(date),
            'news_count': int(len(scores)),
            'weighted_sentiment': float(np.average(scores, weights=conf)),
            'sentiment_volatility': float(np.std(scores)) if len(scores) > 1 else 0.0,
        })
    daily = pd.DataFrame(rows)
    calendar = pd.DataFrame({'Date': pd.date_range(RETRIEVAL_START, DATA_END, freq='D')})
    daily = calendar.merge(daily, on='Date', how='left')
    daily['news_count'] = daily['news_count'].fillna(0).astype(int)
    daily['weighted_sentiment'] = daily['weighted_sentiment'].fillna(0.0)
    daily['sentiment_volatility'] = daily['sentiment_volatility'].fillna(0.0)
    daily['sentiment_momentum_3d'] = daily['weighted_sentiment'].diff(3).fillna(0.0)
    return daily


In [3]:

audit_rows = []
collision_rows = []
daily_series: Dict[str, pd.DataFrame] = {}

for corpus, path in FILES.items():
    daily = daily_finbert(path)
    daily_series[corpus] = daily
    for period, (start, end) in PERIODS.items():
        sub = daily[(daily['Date'] >= start) & (daily['Date'] <= end)].copy()
        no_news = sub['news_count'].eq(0)
        news_days = ~no_news
        audit_rows.append({
            'corpus': corpus,
            'period': period,
            'start_date': start.date().isoformat(),
            'end_date': end.date().isoformat(),
            'calendar_days': int(len(sub)),
            'no_news_days': int(no_news.sum()),
            'no_news_pct': float(100 * no_news.mean()) if len(sub) else np.nan,
            'days_with_news': int(news_days.sum()),
            'mean_articles_per_day': float(sub['news_count'].mean()) if len(sub) else np.nan,
            'median_articles_per_day': float(sub['news_count'].median()) if len(sub) else np.nan,
        })

    # Exact feature-collision diagnostic: all 3 forecasting FinBERT features equal zero.
    sample = daily[(daily['Date'] >= MODEL_START) & (daily['Date'] <= DATA_END)].copy()
    all_zero_features = (
        np.isclose(sample['weighted_sentiment'], 0.0, atol=1e-12)
        & np.isclose(sample['sentiment_volatility'], 0.0, atol=1e-12)
        & np.isclose(sample['sentiment_momentum_3d'], 0.0, atol=1e-12)
    )
    collision_rows.append({
        'corpus': corpus,
        'no_news_days_with_all_three_features_zero': int((sample['news_count'].eq(0) & all_zero_features).sum()),
        'news_days_with_all_three_features_zero': int((sample['news_count'].gt(0) & all_zero_features).sum()),
    })

no_news_audit = pd.DataFrame(audit_rows)
collision_audit = pd.DataFrame(collision_rows)

no_news_audit.to_csv(OUTPUT_DIR / 'no_news_day_audit.csv', index=False)
collision_audit.to_csv(OUTPUT_DIR / 'zero_feature_collision_audit.csv', index=False)

display(no_news_audit)
display(collision_audit)


,corpus,period,start_date,end_date,calendar_days,no_news_days,no_news_pct,days_with_news,mean_articles_per_day,median_articles_per_day
0,general,full_retrieval_window,2022-04-01,2026-08-31,1614,0,0.000000,1614,841.807311,52.0
1,general,forecast_sample,2022-05-01,2026-08-31,1584,0,0.000000,1584,857.231692,53.0
2,general,train_query_dates,2022-05-01,2025-03-31,1066,0,0.000000,1066,42.796435,37.5
3,general,validation_query_dates,2025-04-01,2025-10-31,214,0,0.000000,214,455.542056,200.5
4,general,test_query_dates,2025-11-01,2026-08-31,304,0,0.000000,304,3995.881579,4091.5
5,BTC_specific,full_retrieval_window,2022-04-01,2026-08-31,1614,0,0.000000,1614,38.035316,39.0
6,BTC_specific,forecast_sample,2022-05-01,2026-08-31,1584,0,0.000000,1584,37.609848,39.0
7,BTC_specific,train_query_dates,2022-05-01,2025-03-31,1066,0,0.000000,1066,40.100375,41.0
8,BTC_specific,validation_query_dates,2025-04-01,2025-10-31,214,0,0.000000,214,43.598131,48.0
9,BTC_specific,test_query_dates,2025-11-01,2026-08-31,304,0,0.000000,304,24.661184,22.0


,corpus,no_news_days_with_all_three_features_zero,news_days_with_all_three_features_zero
0,general,0,0
1,BTC_specific,0,0
2,ETH_specific,0,0


In [4]:

feature_input_audit = pd.DataFrame([
    {
        'phase': 'Phase 1 API sentiment',
        'news_count_computed': True,
        'news_count_used_as_forecasting_feature': True,
        'explicit_no_news_indicator': False,
        'note': 'total_news_count is one of the Phase 1 sentiment features.'
    },
    {
        'phase': 'Phase 2 general FinBERT',
        'news_count_computed': True,
        'news_count_used_as_forecasting_feature': False,
        'explicit_no_news_indicator': False,
        'note': 'Forecasting inputs use weighted sentiment, sentiment volatility, and 3-day momentum.'
    },
    {
        'phase': 'Phase 3 crypto-specific FinBERT',
        'news_count_computed': True,
        'news_count_used_as_forecasting_feature': False,
        'explicit_no_news_indicator': False,
        'note': 'Forecasting inputs use weighted sentiment, sentiment volatility, and 3-day momentum.'
    },
    {
        'phase': 'Phase 4 temporal FinBERT',
        'news_count_computed': True,
        'news_count_used_as_forecasting_feature': False,
        'explicit_no_news_indicator': False,
        'note': 'Same three FinBERT features as Phase 3; temporal treatment changes sequence weighting/lookback only.'
    },
    {
        'phase': 'Phase 5 reliability fusion',
        'news_count_computed': True,
        'news_count_used_as_forecasting_feature': False,
        'explicit_no_news_indicator': False,
        'note': 'news_count enters the reliability-weight construction and missing-scope logic, but is not a direct forecasting feature.'
    },
])
feature_input_audit.to_csv(OUTPUT_DIR / 'finbert_feature_input_audit.csv', index=False)
display(feature_input_audit)


,phase,news_count_computed,news_count_used_as_forecasting_feature,explicit_no_news_indicator,note
0,Phase 1 API sentiment,True,True,False,total_news_count is one of the Phase 1 sentime...
1,Phase 2 general FinBERT,True,False,False,"Forecasting inputs use weighted sentiment, sen..."
2,Phase 3 crypto-specific FinBERT,True,False,False,"Forecasting inputs use weighted sentiment, sen..."
3,Phase 4 temporal FinBERT,True,False,False,Same three FinBERT features as Phase 3; tempor...
4,Phase 5 reliability fusion,True,False,False,news_count enters the reliability-weight const...



## How to interpret the audit

- If no-news days are **rare or essentially absent**, no modeling change is necessary. Report the counts if useful and proceed.
- If no-news days occur with nontrivial frequency, the existing Phase 2–4 specification should be described transparently: `news_count` is computed during aggregation but is not among the three FinBERT forecasting features.
- A concise limitation sentence is:

  **“In the FinBERT phases, zero-filled no-news days may be indistinguishable from days with neutral aggregate sentiment because article count is not included among the forecasting features.”**

This notebook is diagnostic only and intentionally does not add a new feature or rerun Phases 2–4.
